In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader

from sklearn.metrics import accuracy_score

from tqdm import tqdm

import matplotlib.pyplot as plt

In [2]:
from pathlib import Path

cwd = Path.cwd().resolve()
possible_roots = [cwd, cwd.parent]

data_dir = None
for root in possible_roots:
    candidate = root / "datasets" / "Chest X-Ray Images" / "chest_xray" / "chest_xray"
    if candidate.exists():
        data_dir = candidate
        break

if data_dir is None:
    raise FileNotFoundError(
        "Could not find the Chest X-Ray dataset. Checked: "
        + ", ".join(str(root / 'datasets' / 'Chest X-Ray Images' / 'chest_xray' / 'chest_xray') for root in possible_roots)
    )

print("Using data_dir:", data_dir)

batch_size = 16
num_workers = 0

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset = datasets.ImageFolder(os.path.join(data_dir, "train"), transform=train_transforms)
val_dataset = datasets.ImageFolder(os.path.join(data_dir, "val"), transform=val_transforms)
test_dataset = datasets.ImageFolder(os.path.join(data_dir, "test"), transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)

print("Train samples:", len(train_dataset))
print("Val samples:", len(val_dataset))
print("Test samples:", len(test_dataset))
print("Classes:", train_dataset.classes)

Using data_dir: C:\MediVisionAI\datasets\Chest X-Ray Images\chest_xray\chest_xray
Train samples: 5216
Val samples: 16
Test samples: 624
Classes: ['NORMAL', 'PNEUMONIA']


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device :", device)

Device : cpu


In [4]:
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

In [5]:
for param in model.parameters():
    param.requires_grad = False

In [6]:
num_features = model.fc.in_features

model.fc = nn.Linear(num_features, 2)

In [7]:
model = model.to(device)

In [8]:
weights = torch.tensor([3850/1340, 1.0], dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(weight=weights)

In [9]:
optimizer = optim.Adam(
    model.fc.parameters(),
    lr=0.001
)

In [10]:
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.1,
    patience=2
)

In [11]:
EPOCHS = 10

train_losses = []
val_losses = []

train_accuracies = []
val_accuracies = []

best_val_accuracy = 0

In [12]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):

    model.train()

    running_loss = 0
    correct = 0
    total = 0

    for images, labels in tqdm(dataloader):

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(dataloader)
    epoch_accuracy = 100 * correct / total

    return epoch_loss, epoch_accuracy

In [13]:
def validate(model, dataloader, criterion, device):

    model.eval()

    running_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in dataloader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)

            correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(dataloader)
    epoch_accuracy = 100 * correct / total

    return epoch_loss, epoch_accuracy

In [14]:
for epoch in range(EPOCHS):

    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    val_loss, val_acc = validate(
        model,
        val_loader,
        criterion,
        device
    )

    scheduler.step(val_loss)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)

    print(f"Train Loss : {train_loss:.4f}")
    print(f"Train Accuracy : {train_acc:.2f}%")

    print(f"Validation Loss : {val_loss:.4f}")
    print(f"Validation Accuracy : {val_acc:.2f}%")

    if val_acc > best_val_accuracy:

        best_val_accuracy = val_acc

        torch.save(model.state_dict(), "best_resnet50.pth")

        print("✅ Best Model Saved!")


Epoch 1/10


  0%|          | 0/326 [00:00<?, ?it/s]c:\MediVisionAI\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
100%|██████████| 326/326 [14:50<00:00,  2.73s/it]


Train Loss : 0.2985
Train Accuracy : 87.96%
Validation Loss : 0.4579
Validation Accuracy : 68.75%
✅ Best Model Saved!

Epoch 2/10


100%|██████████| 326/326 [20:16<00:00,  3.73s/it]


Train Loss : 0.2062
Train Accuracy : 91.74%
Validation Loss : 0.6272
Validation Accuracy : 68.75%

Epoch 3/10


...

Train Loss : 0.1542
Train Accuracy : 94.08%
Validation Loss : 0.7420
Validation Accuracy : 75.00%

Epoch 9/10


100%|██████████| 326/326 [1:11:59<00:00, 13.25s/it]  


Train Loss : 0.1415
Train Accuracy : 94.71%
Validation Loss : 0.5603
Validation Accuracy : 81.25%

Epoch 10/10


100%|██████████| 326/326 [26:44<00:00,  4.92s/it]


Train Loss : 0.1420
Train Accuracy : 94.69%
Validation Loss : 0.6716
Validation Accuracy : 81.25%
